# Challenge 3: Robust Tagging Under Changing Detector Conditions

## What's the challenge?

A particle physics detector maps candidate particles in **η (eta) and φ (phi)** — pseudorapidity and azimuthal angle. When detector elements go offline — dead channels, noisy modules, alignment failures — particles whose trajectories pass through those regions simply don't show up in the event record.

Goal: **train a model whose anomaly detection stays robust as more of the detector drops out.**

At eval time the model sees events where affected candidates are missing (represented as zeros, same as padding). How you train for this is entirely up to you — augmentation strategy, architecture, objective. The only fixed constraints are the input format and how AUC is scored.

---

## Training data

- **Background only**: QCD, Drell-Yan, tt̄, W+jets — what the detector sees most of the time
- Each event: `[N, 7]` tensor of PF candidates with features `(pt, η, φ, dxy, dxy_sig, is_pf, pdgId)`. Padding rows (and missing candidates) have `pt == 0`.
- You're given the preprocessed tensors directly: `pt_files/robust_tagging_train_data.pt`, `pt_files/robust_tagging_eval.pt`.

## Scoring

During evaluation the model will see both background and signal (anomaly) events and outputs a per-event anomaly score. In the eval scipt, evaluation is repeated multiple times for different severity of degradation applied to the evaluation dataset. The success of your model will be determined via **AUC vs. degradation severity** plots. A robust model maintains high AUC as the severity of detector errors increases. Your model's final score is the area under that curve.

---

## What you'll modify

Two files, not this notebook:

- **`src/embedding/degradation.py`** — `Degradation.forward()` is currently a stub ("Add your degradation code here!"). This simulates detector dropout: `train.py` calls it with `severity=None` during training (implement your own randomized augmentation there), and `eval.py` calls it with a fixed `severity` for each step of the AUC-vs-severity sweep. Keep the class signature so both call sites keep working.
- **`src/embedding/models.py`** — the baseline architecture (`TransformerEncoder`, `Projector`, ...) used by `train.py`/`eval.py`. Change layers, swap the encoder, add heads — anything, as long as it still produces a latent embedding.
- **`configs/train_config.yaml`** — hyperparameters (lr, embed size, loss weights, etc.).

## How this project runs

This notebook runs `train.py` and `eval.py` directly on the GPU attached to it, and shows you the resulting AUC-vs-severity plots.

---
## Section 1: Setup

In [ ]:
import glob
import os

import torch
from IPython.display import Image, display

REPO_DIR   = "/hackathon-data/C9_robust_tagging/escheuller-dev"
DATA_DIR   = "/hackathon-data/C9_robust_tagging/pt_files"
CKPT_DIR   = "/hackathon-data/C9_robust_tagging/checkpoints"
EVAL_PLOTS_DIR = "/hackathon-data/C9_robust_tagging/eval_plots"

TRAIN_DATA_CFG = "configs/data_config_collide1m.yaml"
EVAL_DATA_CFG  = "configs/data_config_eval.yaml"
TRAIN_CFG      = "configs/train_config.yaml"

TRAIN_PT = os.path.join(DATA_DIR, "robust_tagging_train_data.pt")
EVAL_PT  = os.path.join(DATA_DIR, "robust_tagging_eval.pt")

os.chdir(REPO_DIR)  # scripts assume this cwd (relative config paths, `embedding` on path via editable install)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

---
## Section 2: Train

`train.py` trains the preprocessor + Transformer encoder + projector + classifier on the background classes (SupCon-style contrastive loss + cross-entropy), saving the best (lowest val loss) checkpoint to `--outdir`. It picks up whatever you changed in `degradation.py` / `models.py` automatically.

Add `--test_mode` to first do a quick pass on ~10% of the data before committing to a full run.

In [ ]:
!python train.py \
    --data_cfg {TRAIN_DATA_CFG} \
    --train_cfg {TRAIN_CFG} \
    --data {TRAIN_PT} \
    --outdir {CKPT_DIR}
    # add --test_mode above for a quick ~10%-of-data sanity check first

ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "*.pth")))
latest_ckpt = ckpts[-1] if ckpts else None
print("latest checkpoint:", latest_ckpt)

---
## Section 3: Evaluate

`eval.py` loads a trained checkpoint, embeds the nominal eval set, trains a linear probe on those embeddings, then sweeps `--num_severities` degradation levels between `--sev_min` and `--sev_max` — applying your `Degradation` at each fixed severity on the fly (no separate degraded-files step) — and re-embeds/re-scores at each one. Plots AUC vs. severity (the robustness curve) plus t-SNE diagnostics to `--outdir`.

Requires a checkpoint from Section 2 — set `latest_ckpt` above (or hardcode a path) before running this.

In [ ]:
assert latest_ckpt is not None, "train a model in Section 2 first (or set latest_ckpt manually)"

!python eval.py \
    --train_cfg {TRAIN_CFG} \
    --data_cfg {EVAL_DATA_CFG} \
    --encoder {latest_ckpt} \
    --data {EVAL_PT} \
    --outdir {EVAL_PLOTS_DIR} \
    --sev_min 0.0 \
    --sev_max 1.0 \
    --num_severities 10 \
    --grace_period 1000 \
    --diagnostics

---
## Section 4: Look at the results

`eval.py` writes plots to `EVAL_PLOTS_DIR` rather than returning them — display them inline here. A flat `auc_vs_severity` curve near 1.0 is the goal; the printed "area under AUC-vs-severity curve" is the robustness score.

In [ ]:
display(Image(filename=os.path.join(EVAL_PLOTS_DIR, "auc_vs_severity.png")))
display(Image(filename=os.path.join(EVAL_PLOTS_DIR, "tsne_latents.png")))
display(Image(filename=os.path.join(EVAL_PLOTS_DIR, "tsne_zero_fraction_diagnostic.png")))